# Fraud Detection Pipeline

## Project Overview
The goal is to build a machine learning model to detect fraudulent bank card transactions and evaluate its usefulness for a bank.

## Data Sources
Synthetic data available in Databricks:
- `transactions` — transaction records.
- `cards` — bank card information.
- `users` — customer information.
- `Fraud labels` — fraud indicators (`Yes` / `No`).

## Tools
- **Databricks:** SQL, Python, and dashboards.
- **Scikit-learn:** model development and evaluation.
- **MLflow:** experiment tracking.
- **GitHub:** version control.

## Workflow
1. Data exploration (EDA).
2. Data cleaning and feature engineering.
3. Time-based data splitting.
4. Model training and evaluation.
5. Threshold optimization.
6. Results and dashboard.

## Expected Outcome
A complete fraud detection pipeline with an evaluated model and a dashboard presenting key results.

# Part 1 — GitHub and Databricks Setup

## Objective
Prepare the project environment and configure version control.

## Completed Tasks
- Created the `fraud-detection-databricks` repository.
- Created a README and the `fraud_detection` notebook.
- Connected GitHub to Databricks.
- Configured Git for version control.

## Conclusion
The project environment is ready for development.

# Part 2 — Data Loading and Exploration (EDA)

## Objective
Explore the dataset structure and identify patterns in fraudulent transactions.

## Tasks
- Inspect tables, columns, data types, and missing values.
- Calculate the overall fraud rate.
- Analyze fraud rates by time, transaction type, and card type.
- Compare fraudulent and normal transaction amounts.
- Document five key findings with numerical evidence.

## Conclusion
Summarize the main insights and patterns identified during data exploration.

In [0]:
%sql
-- Part 2: Data Loading and Exploration (EDA)
-- Step 1: Inspect Transaction Data
-- Objective: Verify that transaction data was loaded correctly.
-- Display the first 10 rows to review column names, data structure, and example values.

SELECT *
FROM payments.default.pay_transactions
LIMIT 10;

In [0]:
%sql
-- Step 2: Inspect Card Data
-- Objective: Explore the card dataset.
-- Display the first 10 rows to review available card attributes and example values.

SELECT *
FROM payments.default.pay_cards
LIMIT 10;

In [0]:
%sql
-- Step 3: Inspect User Data
-- Objective: Explore customer information.
-- Display the first 10 rows to review demographic and financial attributes.

SELECT *
FROM payments.default.pay_users
LIMIT 10;

In [0]:
%sql
-- Step 4: Inspect Fraud Labels
-- Objective: Understand how fraudulent transactions are identified.
-- Display the first 10 rows to inspect transaction IDs and fraud labels.

SELECT *
FROM payments.default.pay_labels
LIMIT 10;

In [0]:
%sql
-- Step 5: Inspect Client Insider Data
-- Objective: Explore customer-level summary information.
-- Review age groups, total spending, and transaction frequency.
-- Note: Aggregated values may cause data leakage if used incorrectly in a prediction model.

SELECT *
FROM payments.default.client_insider
LIMIT 10;

In [0]:
%sql
-- Step 6: Check Table Sizes
-- Objective: Understand the size of each dataset.
-- Count the rows in all five tables to verify the loaded data.

SELECT 'pay_transactions' AS table_name, COUNT(*) AS row_count
FROM payments.default.pay_transactions

UNION ALL

SELECT 'pay_cards', COUNT(*)
FROM payments.default.pay_cards

UNION ALL

SELECT 'pay_users', COUNT(*)
FROM payments.default.pay_users

UNION ALL

SELECT 'pay_labels', COUNT(*)
FROM payments.default.pay_labels

UNION ALL

SELECT 'client_insider', COUNT(*)
FROM payments.default.client_insider;

In [0]:
%sql
-- Step 7.1: Inspect Transaction Schema
-- Objective: Check column names and data types.
-- Identify date, numeric, text, and identifier columns.

DESCRIBE TABLE payments.default.pay_transactions;

In [0]:
%sql
-- Step 7.2: Inspect Card Schema
-- Objective: Check the structure and data types of card columns.
-- Identify attributes that may be useful for further analysis.

DESCRIBE TABLE payments.default.pay_cards;

In [0]:
%sql
-- Step 7.3: Inspect User Schema
-- Objective: Check the structure and data types of customer columns.
-- Verify the data types of age, income, credit score, and other attributes.

DESCRIBE TABLE payments.default.pay_users;

In [0]:
%sql
-- Step 7.4: Inspect Fraud Label Schema
-- Objective: Verify the structure of the fraud labels.
-- Check the transaction identifier and fraud indicator data types.

DESCRIBE TABLE payments.default.pay_labels;

In [0]:
%sql
-- Step 8: Calculate Overall Fraud Rate
-- Objective: Measure the proportion of fraudulent transactions.
-- Join transactions with fraud labels using the correct transaction identifier.
-- IMPORTANT: Replace t.id below if the transaction ID column has a different name.

SELECT
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN l.is_fraud = 'Yes' THEN 1 ELSE 0 END) AS fraud_transactions,
    ROUND(
        100.0 * SUM(CASE WHEN l.is_fraud = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS fraud_rate_pct
FROM payments.default.pay_transactions t
INNER JOIN payments.default.pay_labels l
    ON t.id = l.transaction_id;

In [0]:
%sql
-- Step 9: Create Base View for EDA
-- Objective: One reusable view with the join, the binary target and a proper timestamp.
-- Note: amount is already DOUBLE, so no "$" cleaning is needed.

CREATE OR REPLACE TEMP VIEW eda_base AS
SELECT
    t.id,
    t.client_id,
    t.card_id,
    -- Convert text date to a real timestamp for time-based analysis and splitting.
    CAST(t.date AS TIMESTAMP) AS tx_ts,
    t.amount,
    t.use_chip,
    t.merchant_id,
    t.merchant_city,
    t.merchant_state,
    CAST(t.zip AS STRING) AS zip,    -- zip is a code, not a number
    t.mcc,
    t.errors,
    l.is_fraud,
    -- Binary target: 1 = fraud, 0 = normal. UPPER() makes the match case-insensitive.
    CASE WHEN UPPER(l.is_fraud) = 'YES' THEN 1 ELSE 0 END AS fraud_flag,
    c.card_type
FROM payments.default.pay_transactions t
-- INNER JOIN: keep only labeled transactions.
INNER JOIN payments.default.pay_labels l
    ON t.id = l.transaction_id
-- LEFT JOIN: keep transactions even if the card is missing.
LEFT JOIN payments.default.pay_cards c
    ON t.card_id = c.id;

In [0]:
%sql
-- Step 10: Fraud Rate by Year and Month
-- Objective: Check whether the fraud rate is stable over time.
-- This matters for the time-based split in Part 4.
-- Chart: line chart, x = (yr, mo), y = fraud_rate_pct.

SELECT
    YEAR(date)  AS yr,
    MONTH(date) AS mo,
    COUNT(*)          AS n_tx,        -- total transactions in the month
    SUM(fraud_flag)   AS n_fraud,     -- fraud count (small numbers -> noisy rate)
    ROUND(100.0 * AVG(fraud_flag), 4) AS fraud_rate_pct
FROM eda_base
GROUP BY 1, 2      -- group by the first and second SELECT columns
ORDER BY 1, 2;     -- chronological order

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Step 11.1: Fraud Rate by Hour of Day
-- Objective: Check whether night hours are riskier than daytime.
-- If the date column is text, use HOUR(CAST(date AS TIMESTAMP)).

SELECT
    HOUR(date) AS hr,
    COUNT(*)         AS n_tx,
    SUM(fraud_flag)  AS n_fraud,
    ROUND(100.0 * AVG(fraud_flag), 4) AS fraud_rate_pct
FROM eda_base
GROUP BY 1
ORDER BY 1;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Step 11.2: Fraud Rate by Transaction Type
-- Objective: Compare swipe, chip and online transactions.
-- Compare rates, not counts: groups differ a lot in size.

SELECT
    use_chip  AS tx_type,
    COUNT(*)        AS n_tx,
    SUM(fraud_flag) AS n_fraud,
    ROUND(100.0 * AVG(fraud_flag), 4) AS fraud_rate_pct
FROM eda_base
GROUP BY 1
ORDER BY fraud_rate_pct DESC;   -- riskiest type first

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Step 11.3: Fraud Rate by Card Type
-- Objective: Compare credit, debit and prepaid cards.
-- HAVING filters out tiny groups, where the rate is just noise.

SELECT
    card_type,
    COUNT(*)        AS n_tx,
    SUM(fraud_flag) AS n_fraud,
    ROUND(100.0 * AVG(fraud_flag), 4) AS fraud_rate_pct
FROM eda_base
GROUP BY 1
HAVING COUNT(*) >= 1000         -- minimum group size for a reliable rate
ORDER BY fraud_rate_pct DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Step 12: Transaction Amounts, Fraud vs Normal
-- Objective: Compare the amount distribution of fraudulent and normal transactions.
-- Amounts are right-skewed, so median and p95 are more reliable than the mean.
-- Chart: boxplot by is_fraud, or a histogram on a log scale.
-- Note: negative amounts (refunds) may exist; they are handled in Part 3.

SELECT
    is_fraud,
    COUNT(*)                                   AS n,
    ROUND(AVG(amount_num), 2)                  AS mean_amt,    -- pulled up by outliers
    ROUND(PERCENTILE(amount_num, 0.5), 2)      AS median_amt,  -- typical transaction
    ROUND(PERCENTILE(amount_num, 0.95), 2)     AS p95_amt,     -- upper tail
    ROUND(MAX(amount_num), 2)                  AS max_amt
FROM eda_base
GROUP BY 1;

Databricks visualization. Run in Databricks to view.

%md
## Data Format Notes

Source tables stay unchanged. Fixes are applied in Part 3 when building the clean tables.

| Source table | Column | Current | Fix | Applied in |
|---|---|---|---|---|
| pay_transactions | date | STRING `2010-02-17 06:42:00` | CAST to TIMESTAMP | clean_transactions |
| pay_transactions | zip | DOUBLE | CAST to STRING (it is a code) | clean_transactions |
| pay_cards | expires | STRING `02/2021` | to_date(col, 'MM/yyyy') | clean_cards |
| pay_cards | acct_open_date | STRING `04/2003` | to_date(col, 'MM/yyyy') | clean_cards |
| pay_cards | credit_limit | STRING `$15847` | remove `$`, cast to DOUBLE | clean_cards |
| pay_cards | has_chip | STRING `YES` | UPPER(), map to 1/0 | clean_cards |
| pay_cards | card_on_dark_web | STRING `No` | UPPER(), map to 1/0 | clean_cards |
| pay_users | per_capita_income | STRING `$29278` | remove `$`, cast to DOUBLE | clean_users |
| pay_users | yearly_income | STRING `$59696` | remove `$`, cast to DOUBLE | clean_users |
| pay_users | total_debt | STRING `$127613` | remove `$`, cast to DOUBLE | clean_users |
| pay_labels | is_fraud | STRING `Yes` / `No` | UPPER(), map to 1/0 as fraud_flag | at join with transactions |

### Not changed
- pay_transactions: amount (already DOUBLE), id, client_id, card_id, merchant_id, mcc (IDs and codes)
- pay_users: ages, birth_year, birth_month, credit_score, num_credit_cards, latitude, longitude
- pay_cards: num_cards_issued, year_pin_last_changed

### Notes
- amount has 64,044 negative values (5.2%), likely refunds; decide in Part 3.
- card_number and cvv are sensitive: never used as features.
- client_insider has the same income and debt columns as pay_users, plus total_spending and transaction_frequency, which may be computed over the full period (leakage risk). Not used until checked.

%md
## Key Findings

1. **Base rate:** fraud is rare: 2,573 fraudulent transactions out of 831,529 labeled ones (0.31%). A model that always predicts "not fraud" would be 99.69% accurate and useless, so accuracy is not a suitable metric.
2. **Time (months):** the monthly fraud rate is not stable: it ranges from 0.16% in January and 0.20% in December to 0.41% in February (2010 only).
3. **Time (hours):** fraud peaks at night: 0.96% at 3:00, 1.05% at 4:00 and 1.20% at 5:00, which is 3-4 times the overall rate. From 17:00 to 22:00 it stays below 0.05%.
4. **Transaction type:** online transactions have a fraud rate of 2.57% versus 0.03% for swipe, about 75 times higher. Online is only 11% of transactions but holds 90% of all fraud (2,317 of 2,573). Card type differs little (0.28% debit, 0.34% credit, 0.44% prepaid).
5. **Amounts:** fraudulent transactions are larger: median 74.29 vs 29.40, mean 119.68 vs 43.33, p95 395.23 vs 146.89. Maximums are similar (4,978 vs 4,266), so the difference is in the typical amount, not in outliers.

**Data notes:** labels exist for 831,529 of 1,240,880 transactions (67.0%), spread evenly across months; all transactions are from 2010; there is no "chip" transaction type, only Online and Swipe.

%md
## Ключевые находки

1. **Базовая доля:** фрод редок: 2 573 мошеннические транзакции из 831 529 размеченных (0,31%). Модель, которая всегда предсказывает «не фрод», будет права в 99,69% случаев и бесполезна, поэтому accuracy не подходит как метрика.
2. **Время (месяцы):** доля фрода по месяцам нестабильна: от 0,16% в январе и 0,20% в декабре до 0,41% в феврале (только 2010 год).
3. **Время (часы):** пик фрода приходится на ночь: 0,96% в 3:00, 1,05% в 4:00 и 1,20% в 5:00, то есть в 3-4 раза выше общей доли. С 17:00 до 22:00 доля ниже 0,05%.
4. **Тип транзакции:** у онлайн-транзакций доля фрода 2,57% против 0,03% у свайпа, примерно в 75 раз выше. Онлайн составляет лишь 11% транзакций, но на него приходится 90% всего фрода (2 317 из 2 573). Тип карты различает слабо (0,28% дебетовая, 0,34% кредитная, 0,44% предоплаченная).
5. **Суммы:** мошеннические транзакции типично в 2,5 раза крупнее обычных: медиана 74,29 против 29,40. Самые крупные платежи у обеих групп близки (около 4 000-5 000), поэтому сумма помогает, но один этот признак фрод не отделяет.

**Заметки о данных:** метки есть у 831 529 из 1 240 880 транзакций (67,0%), равномерно по месяцам; все транзакции за 2010 год; типа транзакции «chip» нет, только Online и Swipe.